# Simulation Globale LMTL avec Transport (Numba-accelerated)

Ce notebook configure et exécute une simulation globale du modèle LMTL en utilisant les forçages issus du fichier Zarr fourni.
Le transport (advection et diffusion) est activé avec la version optimisée Numba pour améliorer les performances.

In [ ]:
from datetime import timedelta

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pint
import xarray as xr
from IPython.display import Markdown

from seapopym.blueprint import Blueprint
from seapopym.controller import SimulationConfig, SimulationController
from seapopym.lmtl.configuration import LMTLParams
from seapopym.lmtl.core import (
    compute_day_length,
    compute_gillooly_temperature,
    compute_layer_weighted_mean,
    compute_mortality_tendency,
    compute_production_dynamics,
    compute_production_initialization,
    compute_recruitment_age,
    compute_threshold_temperature,
)
from seapopym.standard.coordinates import Coordinates
from seapopym.transport import (
    BoundaryConditions,
    BoundaryType,
    check_diffusion_stability,
    compute_advection_cfl,
    compute_advection_numba,  # Numba-accelerated version
    compute_diffusion_tendency,
    compute_spherical_cell_areas,
    compute_spherical_dx,
    compute_spherical_dy,
    compute_spherical_face_areas_ew,
    compute_spherical_face_areas_ns,
)

In [ ]:
ureg = pint.get_application_registry()

## 1. Chargement et Préparation des Données

In [ ]:
zarr_path = "/Users/adm-lehodey/Documents/Workspace/Projects/seapopym-message/data/pacific_lmtl_forcings.zarr"
ds = xr.open_zarr(zarr_path)
ds

## 2. Configuration de la Simulation

In [ ]:
# Définition de la période de simulation
# On prend une courte période pour tester (ex: 1 mois)
start_date = "2000-01-01"
end_date = "2005-01-01"
timestep = timedelta(hours=12)  # Pas de temps journalier pour commencer

config = SimulationConfig(
    start_date=start_date,
    end_date=end_date,
    timestep=timestep,
)

# Paramètres LMTL spécifiés
lmtl_params = LMTLParams(
    day_layer=0,
    night_layer=0,
    tau_r_0=10.38 * ureg.days,
    gamma_tau_r=ureg.Quantity(0.11, ureg.degC**-1),
    lambda_0=ureg.Quantity(1 / 150, ureg.day**-1),
    gamma_lambda=ureg.Quantity(0.15, ureg.degC**-1),
    E=ureg.Quantity(0.1668, ureg.dimensionless),
    T_ref=ureg.Quantity(0, ureg.degC),
)

In [ ]:
# Coefficient de diffusion horizontale
D = 100.0  # m²/s

## 3. Préparation des Forçages pour la Simulation

In [ ]:
forcings = ds.sel({Coordinates.T.value: slice(start_date, end_date)})
forcings = forcings[["primary_production", "temperature", "current_u", "current_v"]].load()

# Création des cohortes
cohorts = (np.arange(0, np.ceil(lmtl_params.tau_r_0.magnitude) + 1) * ureg.day).to("second")
cohorts_da = xr.DataArray(
    cohorts.magnitude, dims=["cohort"], name="cohort", attrs={"units": "second"}
)

# Ajout des paramètres statiques aux forçages (pour qu'ils soient accessibles)
forcings = forcings.assign_coords(cohort=cohorts_da)
forcings["dt"] = config.timestep.total_seconds()

# Création du mask océan/terre à partir de la température
# 1 = océan, 0 = terre (NaN dans température)
ocean_mask = xr.where(
    forcings["temperature"].isel({Coordinates.T.value: 0, Coordinates.Z.value: 0}).notnull(),
    1.0,
    0.0,
)
ocean_mask.attrs["units"] = "dimensionless"

# --- Transport Grid Setup ---
# Compute grid metrics
lat = forcings[Coordinates.Y.value]
lon = forcings[Coordinates.X.value]

cell_areas = compute_spherical_cell_areas(lat, lon)
face_areas_ew = compute_spherical_face_areas_ew(lat, lon)
face_areas_ns = compute_spherical_face_areas_ns(lat, lon)

# Compute grid spacings for diffusion
dx = compute_spherical_dx(lat, lon)
dy = compute_spherical_dy(lat, lon)

# Add to forcings
forcings["cell_areas"] = cell_areas
forcings["face_areas_ew"] = face_areas_ew
forcings["face_areas_ns"] = face_areas_ns
forcings["dx"] = dx
forcings["dy"] = dy
forcings["ocean_mask"] = ocean_mask

# Define Boundary Conditions (Periodic X, Closed Y)
boundary_conditions = BoundaryConditions(
    west=BoundaryType.CLOSED,
    east=BoundaryType.CLOSED,
    north=BoundaryType.CLOSED,
    south=BoundaryType.CLOSED,
)


forcings

In [ ]:
# Vérification du mask
print("Vérification du mask océan/terre:")
print(f"Shape du mask: {forcings['ocean_mask'].shape}")
print(f"Valeurs uniques: {np.unique(forcings['ocean_mask'].values)}")
print(f"Nombre de cellules océan: {(forcings['ocean_mask'] == 1).sum().values}")
print(f"Nombre de cellules terre: {(forcings['ocean_mask'] == 0).sum().values}")

# Visualisation du mask
plt.figure(figsize=(12, 6))
forcings["ocean_mask"].plot()
plt.title("Mask Océan/Terre (1=océan, 0=terre)")
plt.show()

## 4. État Initial

In [ ]:
# Création de l'état initial (vide pour commencer)
# Dimensions spatiales
lats = forcings[Coordinates.Y.value]
lons = forcings[Coordinates.X.value]

# Biomasse initiale nulle
biomass_init = xr.DataArray(
    np.zeros((len(lats), len(lons))),
    coords={Coordinates.Y.value: lats, Coordinates.X.value: lons},
    dims=(Coordinates.Y.value, Coordinates.X.value),
    name="Zooplankton/biomass",
)

# Production initiale nulle
production_init = xr.DataArray(
    np.zeros((len(lats), len(lons), len(cohorts))),
    coords={Coordinates.Y.value: lats, Coordinates.X.value: lons, "cohort": cohorts_da},
    dims=(Coordinates.Y.value, Coordinates.X.value, "cohort"),
    name="Zooplankton/production",
)

initial_state = xr.Dataset(
    {"Zooplankton/biomass": biomass_init, "Zooplankton/production": production_init}
)
initial_state

In [ ]:
def configure_model(bp):
    # Wrapper for advection (using Numba-accelerated version)
    def advection_wrapper(state, u, v, cell_areas, face_areas_ew, face_areas_ns, mask):
        return compute_advection_numba(
            state=state,
            u=u,
            v=v,
            cell_areas=cell_areas,
            face_areas_ew=face_areas_ew,
            face_areas_ns=face_areas_ns,
            boundary_conditions=boundary_conditions,
            mask=mask,
        )

    # Wrapper for diffusion
    def diffusion_wrapper(state, dx, dy, mask):
        return compute_diffusion_tendency(
            state=state,
            D=D,
            dx=dx,
            dy=dy,
            boundary_conditions=boundary_conditions,
            mask=mask,
        )

    # Enregistrement des forçages
    # Note: temperature est 4D (time, z, y, x) dans le dataset
    bp.register_forcing(
        "temperature",
        dims=(Coordinates.T.value, Coordinates.Z.value, Coordinates.Y.value, Coordinates.X.value),
        units="degree_Celsius",
    )
    bp.register_forcing(
        "primary_production",
        dims=(Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value),
        units="g/m**2/second",
    )
    bp.register_forcing(
        "ocean_mask", dims=(Coordinates.Y.value, Coordinates.X.value), units="dimensionless"
    )

    bp.register_forcing("dt")
    bp.register_forcing("cohort")
    bp.register_forcing(Coordinates.T.value)
    bp.register_forcing(Coordinates.Y.value)
    bp.register_forcing(
        "current_u",
        dims=(Coordinates.T.value, Coordinates.Z.value, Coordinates.Y.value, Coordinates.X.value),
        units="m/s",
    )
    bp.register_forcing(
        "current_v",
        dims=(Coordinates.T.value, Coordinates.Z.value, Coordinates.Y.value, Coordinates.X.value),
        units="m/s",
    )
    bp.register_forcing("cell_areas", dims=(Coordinates.Y.value, Coordinates.X.value), units="m**2")
    bp.register_forcing("face_areas_ew", dims=(Coordinates.Y.value, Coordinates.X.value), units="m")
    bp.register_forcing("face_areas_ns", dims=(Coordinates.Y.value, Coordinates.X.value), units="m")
    bp.register_forcing("dx", dims=(Coordinates.Y.value, Coordinates.X.value), units="m")
    bp.register_forcing("dy", dims=(Coordinates.Y.value, Coordinates.X.value), units="m")

    bp.register_group(
        group_prefix="Zooplankton",
        units=[
            {
                "func": compute_day_length,
                "output_mapping": {"output": "day_length"},
                "input_mapping": {"latitude": Coordinates.Y.value, "time": Coordinates.T.value},
                "output_units": {"output": "dimensionless"},
            },
            {
                "func": compute_layer_weighted_mean,
                "input_mapping": {"forcing": "temperature"},
                "output_mapping": {"output": "mean_temperature"},
                "output_units": {"output": "degree_Celsius"},
            },
            {
                "func": compute_layer_weighted_mean,
                "input_mapping": {"forcing": "current_u"},
                "output_mapping": {"output": "mean_current_u"},
                "output_units": {"output": "m/s"},
            },
            {
                "func": compute_layer_weighted_mean,
                "input_mapping": {"forcing": "current_v"},
                "output_mapping": {"output": "mean_current_v"},
                "output_units": {"output": "m/s"},
            },
            {
                "func": compute_threshold_temperature,
                "input_mapping": {"temperature": "mean_temperature", "min_temperature": "T_ref"},
                "output_mapping": {"output": "thresholded_temperature"},
                "output_units": {"output": "degree_Celsius"},
            },
            {
                "func": compute_gillooly_temperature,
                "input_mapping": {"temperature": "thresholded_temperature"},
                "output_mapping": {"output": "gillooly_temperature"},
                "output_units": {"output": "degree_Celsius"},
            },
            {
                "func": compute_recruitment_age,
                "input_mapping": {"temperature": "gillooly_temperature"},
                "output_mapping": {"output": "recruitment_age"},
                "output_units": {"output": "second"},
            },
            {
                "func": compute_production_initialization,
                "input_mapping": {"cohorts": "cohort"},
                "output_mapping": {"output": "production_source_npp"},
                "output_tendencies": {"output": "production"},
                "output_units": {"output": "g/m**2/second"},
            },
            {
                "func": compute_production_dynamics,
                "input_mapping": {
                    "cohort_ages": "cohort",
                    "dt": "dt",
                },
                "output_mapping": {
                    "production_tendency": "production_tendency",
                    "recruitment_source": "biomass_source",
                },
                "output_tendencies": {
                    "production_tendency": "production",
                    "recruitment_source": "biomass",
                },
                "output_units": {
                    "production_tendency": "g/m**2/second",
                    "recruitment_source": "g/m**2/second",
                },
            },
            {
                "func": compute_mortality_tendency,
                "input_mapping": {"temperature": "gillooly_temperature"},
                "output_mapping": {"mortality_loss": "biomass_mortality"},
                "output_tendencies": {"mortality_loss": "biomass"},
                "output_units": {"mortality_loss": "g/m**2/second"},
            },
            {
                "func": advection_wrapper,
                "input_mapping": {
                    "state": "biomass",
                    "u": "mean_current_u",
                    "v": "mean_current_v",
                    "cell_areas": "cell_areas",
                    "face_areas_ew": "face_areas_ew",
                    "face_areas_ns": "face_areas_ns",
                    "mask": "ocean_mask",
                },
                "output_mapping": {"advection_rate": "advection_tendency"},
                "output_tendencies": {"advection_rate": "biomass"},
                "output_units": {"advection_rate": "g/m**2/second"},
            },
            {
                "func": advection_wrapper,
                "input_mapping": {
                    "state": "production",
                    "u": "mean_current_u",
                    "v": "mean_current_v",
                    "cell_areas": "cell_areas",
                    "face_areas_ew": "face_areas_ew",
                    "face_areas_ns": "face_areas_ns",
                    "mask": "ocean_mask",
                },
                "output_mapping": {"advection_rate": "production_advection_tendency"},
                "output_tendencies": {"advection_rate": "production"},
                "output_units": {"advection_rate": "g/m**2/second"},
            },
            {
                "func": diffusion_wrapper,
                "input_mapping": {
                    "state": "biomass",
                    "dx": "dx",
                    "dy": "dy",
                    "mask": "ocean_mask",
                },
                "output_mapping": {"diffusion_rate": "diffusion_tendency"},
                "output_tendencies": {"diffusion_rate": "biomass"},
                "output_units": {"diffusion_rate": "g/m**2/second"},
            },
            {
                "func": diffusion_wrapper,
                "input_mapping": {
                    "state": "production",
                    "dx": "dx",
                    "dy": "dy",
                    "mask": "ocean_mask",
                },
                "output_mapping": {"diffusion_rate": "production_diffusion_tendency"},
                "output_tendencies": {"diffusion_rate": "production"},
                "output_units": {"diffusion_rate": "g/m**2/second"},
            },
        ],
        parameters={
            "day_layer": {"units": "dimensionless"},
            "night_layer": {"units": "dimensionless"},
            "tau_r_0": {"units": "second"},
            "gamma_tau_r": {"units": "1/degree_Celsius"},
            "lambda_0": {"units": "1/second"},
            "gamma_lambda": {"units": "1/degree_Celsius"},
            "T_ref": {"units": "degree_Celsius"},
            "E": {"units": "dimensionless"},
        },
        state_variables={
            "production": {
                "dims": (Coordinates.Y.value, Coordinates.X.value, "cohort"),
                "units": "g/m**2/second",
            },
            "biomass": {
                "dims": (Coordinates.Y.value, Coordinates.X.value),
                "units": "g/m**2",
            },
        },
    )


bp = Blueprint()
configure_model(bp)
Markdown("```mermaid\n" + bp.export_mermaid() + "\n```")

## 6. Exécution

## 5.5. Vérification de la Stabilité

In [ ]:
# Vérification de la stabilité de la diffusion
stability = check_diffusion_stability(
    D=D,
    dx=forcings["dx"],
    dy=forcings["dy"],
    dt=config.timestep.total_seconds(),
)

print("Vérification de stabilité de la diffusion:")
print(f"  Coefficient de diffusion D = {D} m²/s")
print(f"  Timestep dt = {config.timestep.total_seconds()} s ({config.timestep})")
print(f"  dx min = {stability['dx_min']:.1f} m")
print(f"  dy min = {stability['dy_min']:.1f} m")
print(f"  dt max (stable) = {stability['dt_max']:.1f} s ({stability['dt_max'] / 3600:.2f} heures)")
print(f"  CFL diffusion = {stability['cfl_diffusion']:.4f} (doit être ≤ 0.25)")
print(f"  Marge de sécurité = {stability['margin']:.2f}x")
print(f"  Stable? {'✓ OUI' if stability['is_stable'] else '✗ NON'}")

if not stability["is_stable"]:
    raise ValueError(
        f"La diffusion est instable! Réduisez dt à {stability['dt_max']:.1f} s "
        f"ou réduisez D à {D * stability['margin']:.1f} m²/s"
    )

In [ ]:
controller = SimulationController(config, backend="sequential")
controller.setup(
    configure_model,
    initial_state,
    forcings,
    parameters={"Zooplankton": lmtl_params},
    output_variables=["Zooplankton/biomass"],
)

print("Démarrage de la simulation...")
controller.run()
print("Simulation terminée.")

## 7. Visualisation

In [ ]:
results = controller.results

import matplotlib.pyplot as plt
import numpy as np

num_time_steps = results["Zooplankton/biomass"].time.size
# selected_indices = np.linspace(0, num_time_steps - 1, 8, dtype=int)
start = 300
selected_indices = np.arange(start, start + 7 * 8, 7, dtype=int)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, idx in enumerate(selected_indices):
    time_val = results["Zooplankton/biomass"].time.isel(time=idx).item()
    results["Zooplankton/biomass"].isel(time=idx).plot(
        ax=axes[i], cmap="viridis", add_colorbar=False, vmax=1
    )
    # axes[i].set_title(f"Jour {idx} ({time_val.strftime('%Y-%m-%d')})")
    axes[i].set_xlabel("")
    axes[i].set_ylabel("")

plt.tight_layout()
plt.show()

# plt.figure(figsize=(12, 6))
# results["Zooplankton/biomass"].isel(time=350).plot()
# plt.title(f"Biomasse Zooplankton à la fin de la simulation ({end_date})")
# plt.show()

In [ ]:
results["Zooplankton/biomass"].rename("biomass").to_zarr(
    f"./zooplankton_numba_transport_pacific_seapopym_v2.zarr", mode="w"
)